In [1]:
from qdrant_client import QdrantClient
client = QdrantClient(":memory:")
client 

In [3]:
from qdrant_client.models import Distance,VectorParams
client.create_collection(
    collection_name="my_documents",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)
print("Collection Created")

Collection Created


In [5]:
from sentence_transformers import SentenceTransformer 
model=SentenceTransformer("all-MiniLM-L6-v2")
documents = [
    "I love programming.",
    "I really enjoy coding.",
    "Programming is my favorite thing."
]
documents_embeddings=model.encode(documents)
documents_embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(3, 384)

In [6]:
from qdrant_client.models import PointStruct
points=[
    PointStruct(
        id=1,
        vector=documents_embeddings[0].tolist(),
        payload={
            "document":documents[0],
            "topic":"Machine Learning",
            "language":"python"
        }
    ),

    PointStruct(
        id=2,
        vector=documents_embeddings[1].tolist(),
        payload={
            "document":documents[1],
            "topic":"backend",
            "language":"java"
        }
    ),

    PointStruct(
        id=3,
        vector=documents_embeddings[2].tolist(),
        payload={
            "document":documents[2],
            "topic":"programming",
            "language":"python"
        }
    )
]
client.upsert(
    collection_name="my_documents",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [9]:
ans = client.retrieve(
    collection_name="my_documents",
    ids=[1,2,3]
)
for point in ans:
    print(point)

id=1 payload={'document': 'I love programming.', 'topic': 'Machine Learning', 'language': 'python'} vector=None shard_key=None order_value=None
id=2 payload={'document': 'I really enjoy coding.', 'topic': 'backend', 'language': 'java'} vector=None shard_key=None order_value=None
id=3 payload={'document': 'Programming is my favorite thing.', 'topic': 'programming', 'language': 'python'} vector=None shard_key=None order_value=None


In [10]:
query="I love coding"
query_embedding=model.encode(query)


In [11]:
ans=client.query_points(
    collection_name="my_documents",
    query=query_embedding.tolist(),
    limit=2
)
ans

QueryResponse(points=[ScoredPoint(id=2, version=0, score=0.923307169289534, payload={'document': 'I really enjoy coding.', 'topic': 'backend', 'language': 'java'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=0, score=0.8318350696598136, payload={'document': 'I love programming.', 'topic': 'Machine Learning', 'language': 'python'}, vector=None, shard_key=None, order_value=None)])

In [13]:
from qdrant_client.models import Filter,FieldCondition,MatchValue
filter_condition = Filter(
    must=[
        FieldCondition(
            key="language",
            match=MatchValue(value="java")
        )
    ]
)
filter_condition

Filter(should=None, min_should=None, must=[FieldCondition(key='language', match=MatchValue(value='java'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)], must_not=None)

In [14]:
results = client.query_points(
    collection_name="my_documents",
    query=query_embedding.tolist(),
    query_filter=filter_condition,
    limit=2
)

print(results)

points=[ScoredPoint(id=2, version=0, score=0.9233072457676588, payload={'document': 'I really enjoy coding.', 'topic': 'backend', 'language': 'java'}, vector=None, shard_key=None, order_value=None)]


In [15]:
for point in results.points:
    print("ID:", point.id)
    print("Document:", point.payload["document"])
    print("Score:", point.score)
    print("Language:", point.payload["language"])

ID: 2
Document: I really enjoy coding.
Score: 0.9233072457676588
Language: java
